In [2]:
# Install first:
!pip install ppscore

import pandas as pd
import ppscore as pps


# ---------------------------------------------------------
# 1. Example dataset
# ---------------------------------------------------------

df = pd.DataFrame({
    "Revenue_Growth": [5, 8, 12, 15, 18, 20, 25, 28, 30, 35],
    "ROE":            [8, 10, 12, 14, 16, 18, 20, 22, 24, 26],
    "Debt_Equity":    [2.5, 2.2, 2.0, 1.8, 1.7, 1.5, 1.3, 1.2, 1.0, 0.8],
    "Marketing":      [10, 12, 15, 17, 20, 23, 25, 28, 30, 35],
    "Stock_Return":   [3, 5, 8, 10, 12, 15, 18, 20, 22, 25]
})


# ---------------------------------------------------------
# 2. Define target variable
# ---------------------------------------------------------

target = "Stock_Return"


# ---------------------------------------------------------
# 3. Calculate PPS for every feature
# ---------------------------------------------------------

pps_results = []

for feature in df.columns:

    if feature == target:
        continue

    result = pps.score(
        df,
        x=feature,
        y=target
    )

    pps_results.append({
        "Feature": feature,
        "PPS": result["ppscore"]
    })


# ---------------------------------------------------------
# 4. Create results DataFrame
# ---------------------------------------------------------

pps_df = pd.DataFrame(pps_results)

pps_df = pps_df.sort_values(
    by="PPS",
    ascending=False
)

print(pps_df)

  Preparing metadata (setup.py) ... done
  Created wheel for ppscore: filename=ppscore-1.3.1-py2.py3-none-any.whl size=13180 sha256=30d9d92c2d9d1c4c6450232c8110e06e4aaadb241014eb23a38a47e8133fec90
  Stored in directory: /root/.cache/pip/wheels/84/0d/d6/ec295c574356939e9877f87c4eb195ba97440b0cd97cadf32a
Successfully built ppscore
          Feature       PPS
0  Revenue_Growth  0.630376
3       Marketing  0.623656
2     Debt_Equity  0.610215
1             ROE  0.603495


In [3]:
def pps_decision(ppscore, threshold=0.30):

    if ppscore >= threshold:
        return "SELECT"

    else:
        return "REJECT"


pps_df["Decision"] = pps_df["PPS"].apply(
    pps_decision
)

print(pps_df)

          Feature       PPS Decision
0  Revenue_Growth  0.630376   SELECT
3       Marketing  0.623656   SELECT
2     Debt_Equity  0.610215   SELECT
1             ROE  0.603495   SELECT


Complete automated feature-selection function

You can turn this into a reusable function for your ML/Agentic AI project:

In [6]:
import pandas as pd
import ppscore as pps


def select_features_using_pps(
    df,
    target,
    threshold=0.30
):

    results = []

    for feature in df.columns:

        # Don't calculate PPS of target against itself
        if feature == target:
            continue

        try:

            result = pps.score(
                df,
                x=feature,
                y=target
            )

            score = result["ppscore"]

            if score >= threshold:
                decision = "SELECT"
            else:
                decision = "REJECT"

            results.append({
                "Feature": feature,
                "PPS": score,
                "Decision": decision
            })

        except Exception as e:

            results.append({
                "Feature": feature,
                "PPS": None,
                "Decision": f"ERROR: {e}"
            })

    results_df = pd.DataFrame(results)

    # Rank features
    results_df = results_df.sort_values(
        by="PPS",
        ascending=False
    )

    return results_df

In [7]:
results = select_features_using_pps(
    df,
    target="Stock_Return",
    threshold=0.30
)

print(results)

          Feature       PPS Decision
0  Revenue_Growth  0.630376   SELECT
3       Marketing  0.623656   SELECT
2     Debt_Equity  0.610215   SELECT
1             ROE  0.603495   SELECT


In [8]:
selected_features = results.loc[
    results["Decision"] == "SELECT",
    "Feature"
].tolist()

print("Selected features:")
print(selected_features)

Selected features:
['Revenue_Growth', 'Marketing', 'Debt_Equity', 'ROE']
